# Day 3 · Lab 2 — Circuit Breaker + Retry with Tenacity

## What you'll build

1. A **flaky mock downstream** — random 500s, timeouts, slow responses
2. A **tenacity retry wrapper** — 4 attempts, exponential backoff 1s → 10s
3. A **CircuitBreaker** class — CLOSED → OPEN → HALF-OPEN state machine
4. Combined behavior demonstration: retry rides on top of the breaker

## Prerequisites

- Lab 1 completed
- `tenacity` installed (`pip install --user tenacity`)

## Step 1 — Environment

In [ ]:
import os, sys, subprocess, time, random, asyncio
from pathlib import Path

for pkg in ["tenacity"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--user", pkg])

from tenacity import (
    retry, stop_after_attempt, wait_exponential,
    retry_if_exception_type, before_sleep_log,
)
import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("day3lab2")

print("✓ Ready")

## Step 2 — Flaky mock downstream

Simulates a real SaaS API that fails intermittently. Fails 60% of the time with either a timeout or a 500.

In [ ]:
class DownstreamError(Exception):
    pass


class TimeoutError_(Exception):
    pass


class FlakyDownstream:
    def __init__(self, failure_rate: float = 0.6):
        self.failure_rate = failure_rate
        self.call_count = 0
        self.healthy = True

    async def call(self, payload: str) -> dict:
        self.call_count += 1
        if not self.healthy:
            raise DownstreamError("500 Internal Server Error (service unhealthy)")
        if random.random() < self.failure_rate:
            if random.random() < 0.5:
                raise TimeoutError_("Request timed out after 5s")
            raise DownstreamError("500 Internal Server Error")
        return {"payload": payload, "status": "ok", "call": self.call_count}


downstream = FlakyDownstream(failure_rate=0.6)

# Quick sanity check
random.seed(42)
for i in range(3):
    try:
        r = asyncio.run(downstream.call(f"test-{i}"))
        print(f"  call {i}: OK — {r}")
    except Exception as e:
        print(f"  call {i}: FAIL — {type(e).__name__}: {e}")

## Step 3 — Retry with tenacity: 4 attempts, exponential backoff

Retries on TimeoutError and DownstreamError only. Never retries on other exceptions (e.g. permission errors). Caps at 4 attempts and 10s per wait.

In [ ]:
random.seed(0)   # reproducible


@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type((TimeoutError_, DownstreamError)),
    reraise=True,
    before_sleep=before_sleep_log(log, logging.INFO),
)
async def call_with_retry(payload: str) -> dict:
    return await downstream.call(payload)


# Try 3 requests. Some succeed on first try, some after retries.
for i in range(3):
    try:
        result = asyncio.run(call_with_retry(f"req-{i}"))
        print(f"  req-{i}: SUCCESS after {downstream.call_count} total downstream calls — {result}")
    except Exception as e:
        print(f"  req-{i}: EXHAUSTED after {downstream.call_count} total downstream calls — {type(e).__name__}")

## Step 4 — Circuit breaker state machine

CLOSED → OPEN after 5 failures. OPEN rejects immediately for 30s. HALF-OPEN lets one request through — success closes it, failure re-opens.

In [ ]:
class CircuitOpenError(Exception):
    pass


class CircuitBreaker:
    def __init__(self, failure_threshold: int = 5, cooldown_sec: float = 30.0):
        self.failure_threshold = failure_threshold
        self.cooldown_sec = cooldown_sec
        self.state = "CLOSED"        # CLOSED | OPEN | HALF_OPEN
        self.failures = 0
        self.opened_at = 0.0

    async def call(self, fn, *args, **kwargs):
        # State check before the call
        if self.state == "OPEN":
            if time.time() - self.opened_at > self.cooldown_sec:
                self.state = "HALF_OPEN"
                print(f"  [breaker] cooldown elapsed → HALF_OPEN")
            else:
                raise CircuitOpenError(f"circuit OPEN for {self.cooldown_sec - (time.time() - self.opened_at):.1f}s more")

        # Execute
        try:
            result = await fn(*args, **kwargs)
        except Exception as e:
            self._on_failure()
            raise
        else:
            self._on_success()
            return result

    def _on_success(self):
        if self.state == "HALF_OPEN":
            print(f"  [breaker] HALF_OPEN test succeeded → CLOSED")
        self.state = "CLOSED"
        self.failures = 0

    def _on_failure(self):
        self.failures += 1
        if self.state == "HALF_OPEN":
            print(f"  [breaker] HALF_OPEN test failed → OPEN")
            self.state = "OPEN"
            self.opened_at = time.time()
        elif self.failures >= self.failure_threshold:
            print(f"  [breaker] {self.failures} failures → OPEN")
            self.state = "OPEN"
            self.opened_at = time.time()


breaker = CircuitBreaker(failure_threshold=3, cooldown_sec=5.0)   # short cooldown for demo
print("✓ CircuitBreaker configured (threshold=3, cooldown=5s)")

## Step 5 — Combine retry + breaker

Retry rides ON TOP of the breaker. Order matters: tenacity wraps the breaker.call().

For this demo, we make the downstream reliably unhealthy so the breaker trips.

In [ ]:
downstream.healthy = False   # every call will 500
downstream.call_count = 0


async def call_via_breaker(payload):
    return await breaker.call(downstream.call, payload)


# Send requests until breaker trips
for i in range(6):
    try:
        r = asyncio.run(call_via_breaker(f"unhealthy-{i}"))
        print(f"  req-{i}: SUCCESS — {r}")
    except CircuitOpenError as e:
        print(f"  req-{i}: FAST FAIL — {e}")
    except Exception as e:
        print(f"  req-{i}: DOWNSTREAM FAIL — {type(e).__name__}")

print(f"\nDownstream call_count: {downstream.call_count}")
print(f"Breaker state:         {breaker.state}")

## Step 6 — Recovery: downstream heals, breaker recovers

Wait past cooldown, restore downstream health, watch HALF_OPEN → CLOSED.

In [ ]:
print(f"Waiting {breaker.cooldown_sec}s for cooldown...")
time.sleep(breaker.cooldown_sec + 0.5)

downstream.healthy = True
print("→ Downstream restored to healthy")

# First call: HALF_OPEN test
try:
    r = asyncio.run(call_via_breaker("recovery-test"))
    print(f"  Recovery: SUCCESS — {r}")
except Exception as e:
    print(f"  Recovery: FAIL — {e}")

print(f"\nBreaker state: {breaker.state}")

## Step 7 — Production pattern: MCP tool with retry + breaker

Real Day 3 shape — an MCP-style tool that a LangGraph agent calls, with all resilience built in.

In [ ]:
# Reset for clean demo
downstream = FlakyDownstream(failure_rate=0.3)
breaker = CircuitBreaker(failure_threshold=5, cooldown_sec=30.0)
random.seed(1)


@retry(
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1, min=1, max=10),
    retry=retry_if_exception_type((TimeoutError_, DownstreamError)),
    reraise=True,
)
async def _call_downstream(payload):
    return await downstream.call(payload)


async def mcp_tool(payload: str) -> dict:
    """MCP-style tool: retry INSIDE the breaker. Breaker guards against sustained failure."""
    return await breaker.call(_call_downstream, payload)


# Demo run
for i in range(5):
    try:
        r = asyncio.run(mcp_tool(f"prod-{i}"))
        print(f"  req-{i}: OK  (downstream call_count now {downstream.call_count})")
    except CircuitOpenError as e:
        print(f"  req-{i}: BREAKER OPEN — {e}")
    except Exception as e:
        print(f"  req-{i}: FINAL FAIL — {type(e).__name__}")

print(f"\nBreaker state: {breaker.state}, failures: {breaker.failures}")

## What you learned

1. **Tenacity retry**: exponential backoff, capped attempts, selective retry-on-exception
2. **Circuit breaker state machine**: CLOSED → OPEN (on repeated failure) → HALF-OPEN (one probe) → CLOSED (recovery)
3. **Order matters**: retry INSIDE breaker. Breaker sees each retry attempt as one call, but breaker's counter only ticks on the outer failure.
4. **Recovery**: cooldown expiry + one successful probe closes the circuit
5. **Real MCP shape**: tool = breaker.call(retry-wrapped fn). Agent stays clean.

## Production notes

- **Circuit breaker per downstream, not global.** Each SaaS gets its own breaker.
- **Metric everything.** Failures, breaker state transitions, retry counts. Feeds Day 4's observability.
- **Cooldown depends on downstream SLA.** 30s is typical. Longer for known-slow-to-recover systems.
- **HALF-OPEN test is idempotent-safe.** If your call has side effects, use a read-only probe instead.

## Congratulations

You've completed Day 3. Full enterprise MCP integration pattern with auth + retry + breaker.